**This notebook does the following things:**
* Works on the dataset following AI classification round 1 is done (i.e. for animal type, dog type, complaint broad type).
* Cleans the dates
* Removes duplicates
* Creates a dataset of only stray dogs by removing other animals and pet dogs.
* Post this, this dataset goes through another round of AI classification for sub harm types to dogs. That's in the folder AI_Classification_2.

In [59]:
import pandas as pd
from rapidfuzz import fuzz

In [2]:
df = pd.read_csv("national_with_ai_labels.csv")

In [5]:
# checking numbers of rows
len(df)

1122

In [6]:
# checking column names
df.columns

Index(['sr_no', 'date_of_complaint', 'complaint_number',
       'complainant_name_org', 'category_of_complaint', 'complaint',
       'place_of_incident', 'violation_of_law', 'action_initiated', 'status',
       'date_of_action_taken', 'state_id', 'page', 'ai_animal_type',
       'ai_dog_type', 'ai_complaint_type'],
      dtype='str')

In [8]:
df.shape

(1122, 16)

In [11]:
# checking animal types
df["ai_animal_type"].value_counts(dropna=False, normalize=True)*100
# 70% of the data is about dogs

ai_animal_type
dog        70.231729
unclear     9.714795
cat         8.288770
cow         7.575758
bird        2.673797
horse       1.069519
sheep       0.445633
Name: proportion, dtype: float64

In [15]:
# checking stray dog proportion in the dog dataset
df[df["ai_animal_type"] == "dog"]["ai_dog_type"].value_counts(dropna=False, normalize=True)*100
# 62% of the data is about stray dogs

ai_dog_type
stray             62.436548
pet               30.203046
unclear            7.233503
Not Applicable     0.126904
Name: proportion, dtype: float64

In [17]:
df[df["ai_dog_type"] == "stray"]["ai_complaint_type"].value_counts(dropna=False, normalize=True)*100

ai_complaint_type
harm_or_threat_to_dogs                58.417850
harm_or_threat_to_dogs_and_feeders    24.340771
harm_or_threat_to_feeders              8.924949
harm_fear_or_nuisance_by_dogs          6.490872
harm_due_to_fear                       1.419878
Not Applicable                         0.202840
unclear_or_other                       0.202840
Name: proportion, dtype: float64

#### Formatting Dates

In [18]:
#checking dates for top 20 entries to understand format
df["date_of_complaint"].head(20)

0     12/04/2026
1     17/03/2026
2     12/03/2026
3     04/02/2026
4     11/01/2026
5     02/01/2026
6     30/12/2025
7     07/12/2025
8     04/12/2025
9     24/11/2025
10    19/11/2025
11    18/11/2025
12    15/11/2025
13    09/11/2025
14    06/11/2025
15    05/11/2025
16    04/11/2025
17    13/10/2025
18    11/10/2025
19    07/10/2025
Name: date_of_complaint, dtype: str

In [19]:
#checking dates random 20 entries to understand format
df["date_of_complaint"].sample(20, random_state=42)

970     26/01/2025
823     04/05/2025
96      03/07/2024
716     08/04/2024
1063    06/09/2025
824     14/04/2025
381     15/04/2025
1086    19/03/2025
156     11/08/2025
630     07/04/2026
526     24/06/2025
1008    18/06/2024
309     01/06/2025
912     03/09/2025
667     16/11/2023
270     24/08/2023
1000    22/07/2024
477     09/11/2025
208     03/04/2026
298     03/11/2025
Name: date_of_complaint, dtype: str

In [21]:
#checking value_counts of most common entries to understand format
df["date_of_complaint"].value_counts(ascending=False).head(20)

date_of_complaint
04/08/2024    8
04/11/2025    7
10/06/2025    7
11/10/2025    6
22/08/2025    6
19/08/2025    6
02/04/2025    6
27/03/2025    6
18/03/2025    6
13/02/2025    6
30/07/2025    6
09/01/2025    6
14/08/2025    6
03/06/2025    6
21/03/2025    6
04/01/2025    6
02/05/2025    5
29/04/2025    5
29/03/2025    5
11/01/2025    5
Name: count, dtype: int64

In [23]:
# dates look consistently likee date format: DD/MM/YYYY. 

In [25]:
# converting to date-time to YYYY-MM-DD
df["date_of_complaint_clean"] = pd.to_datetime( #creating new column
  df["date_of_complaint"],
    dayfirst=True, #day comes before month. last = year pandas understands automatically
    errors="coerce" #If pandas finds a value it cannot convert into a date, do not crash. Turn it into NaT.
)

In [26]:
# checking if conversion worked: comparing old and new dates 
df[["date_of_complaint", "date_of_complaint_clean"]].head(20)

,date_of_complaint,date_of_complaint_clean
0,12/04/2026,2026-04-12
1,17/03/2026,2026-03-17
2,12/03/2026,2026-03-12
3,04/02/2026,2026-02-04
4,11/01/2026,2026-01-11
5,02/01/2026,2026-01-02
6,30/12/2025,2025-12-30
7,07/12/2025,2025-12-07
8,04/12/2025,2025-12-04
9,24/11/2025,2025-11-24


In [27]:
# confirming data type is correct
df["date_of_complaint_clean"].dtype

dtype('<M8[us]')

In [28]:
# checking failed dates
df["date_of_complaint_clean"].isna().sum()

np.int64(0)

In [35]:
# checking range 
df["date_of_complaint_clean"].min(), df["date_of_complaint_clean"].max()
# dataset starts from 27th february 2023 and ends at 17th may 2026

(Timestamp('2023-03-27 00:00:00'), Timestamp('2026-05-17 00:00:00'))

In [29]:
# creating year column and calling it "complaint_year"
df["complaint_year"] = df["date_of_complaint_clean"].dt.year

In [33]:
# checking trend of stray dog complaints over years
df[df["ai_dog_type"] == "stray"]["complaint_year"].value_counts()
# complaints have increased, phew!

complaint_year
2025    309
2024    122
2026     35
2023     27
Name: count, dtype: int64

### Creating stray dog dataset

In [38]:
df_stray = df[
    (df["ai_animal_type"] == "dog") &
    (df["ai_dog_type"] == "stray")
]

In [39]:
len(df_stray)

492

#### Cleaning

In [52]:
# check whether entire rows are duplicated:
df_stray.duplicated().sum()
# no entire rows are duplicated

np.int64(0)

In [54]:
# # check whether complaint numbers are duplicated:
df_stray["complaint_number"].duplicated().sum()

np.int64(0)

In [68]:
#cleaning complaint text to check duplicates again
df_stray["complaint_clean"] = (
    df_stray["complaint"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df_stray["complaint_clean"].duplicated(keep=False).sum()

np.int64(10)

In [69]:
# checking exact duplicates
exact_text_duplicates = df_stray[
    df_stray["complaint_clean"].duplicated(keep=False)
].sort_values("complaint_clean")

exact_text_duplicates[
    ["complaint_number", "date_of_complaint", "state_id", "status", "complaint"]
]

,complaint_number,date_of_complaint,state_id,status,complaint
311,9-6/2025-26/1760/PCA,07/05/2025,28,-,Dear Sir\r\n Bhilwara Municipal coporation has...
312,9-6/2025-26/1738/PCA,01/05/2025,28,-,Dear Sir\r\nBhilwara Municipal coporation has ...
371,9-14/2025-26/1940/PCA,21/06/2025,15,-,"Feeders not allowed to Feed Stray Pups, have b..."
372,9-14/2025-26/1939/PCA,21/06/2025,15,-,"Feeders not allowed to Feed Stray Pups, have b..."
479,9-3/2025-26/2519/PCA,29/10/2025,35,-,I wish to lodge a formal complaint regarding a...
480,9-3/2025-26/2518/PCA,29/10/2025,35,-,I wish to lodge a formal complaint regarding a...
1068,9-13/2025-26/2088/PCA,23/07/2025,36,-,"Myself Subho Bhowmick, in my area we face seri..."
1069,9-13/2025-26/2077/PCA,21/07/2025,36,-,"Myself Subho Bhowmick, in my area we face seri..."
256,9-5/2024-25/1403/PCA,04/02/2025,8,-,We are seeking your kind attention to the crue...
260,9-5/2024-25/1307/PCA,03/02/2025,8,-,We are seeking your kind attention to the crue...


In [72]:
#removing exact duplicates 
df_stray_clean = df_stray.drop_duplicates(subset="complaint_clean", keep="first")

In [73]:
len(df_stray_clean)

487

Cleaning methodology 

I removed exact duplicate complaint texts after normalizing whitespace and capitalization. I did not systematically remove near-duplicates because similar wording may represent repeated complaints, related incidents, or multiple complainants reporting the same conflict.

Even if two complaints are similar, they may still be valid separate complaints if:
two people complained about the same issue
the same conflict continued over time
a complaint was refiled with more detail
the same person submitted a follow-up
different dogs/feeders were involved in the same area

Fuzzy matching cannot reliably tell those apart.

In [99]:
df_stray_clean.to_csv("data_stray_dog_complaints.csv", index = False)

In [76]:
df_stray["ai_complaint_type"].value_counts(normalize=True)*100

ai_complaint_type
harm_or_threat_to_dogs                58.536585
harm_or_threat_to_dogs_and_feeders    24.390244
harm_or_threat_to_feeders              8.943089
harm_fear_or_nuisance_by_dogs          6.300813
harm_due_to_fear                       1.422764
Not Applicable                         0.203252
unclear_or_other                       0.203252
Name: proportion, dtype: float64

In [77]:
# checking trend of stray dog complaints over years by complaint type
complaint_type_pct_by_year = (
    df_stray
    .groupby("complaint_year")["ai_complaint_type"]
    .value_counts(normalize=True)
    .mul(100)
    .unstack()
)

complaint_type_pct_by_year

ai_complaint_type,Not Applicable,harm_due_to_fear,harm_fear_or_nuisance_by_dogs,harm_or_threat_to_dogs,harm_or_threat_to_dogs_and_feeders,harm_or_threat_to_feeders,unclear_or_other
complaint_year,,,,,,,
2023,NaN,3.703704,NaN,59.259259,29.629630,7.407407,NaN
2024,NaN,0.819672,8.196721,56.557377,26.229508,7.377049,0.819672
2025,0.324675,1.298701,6.493506,59.090909,23.051948,9.740260,NaN
2026,NaN,2.857143,2.857143,60.000000,25.714286,8.571429,NaN


In [98]:
# is harm to dog / feeders increasing / decreasing over time?
(
    df_stray
    .groupby("complaint_year")["harm_direction"]
    .value_counts()
    .mul(100)
    .unstack()
)

harm_direction,harm_by_dogs,harm_to_dogs,other
complaint_year,,,
2023,NaN,2700.0,NaN
2024,1000.0,11100.0,100.0
2025,2000.0,28700.0,100.0
2026,100.0,3400.0,NaN
